# 00 - Transpiler pipeline: từ mạch lý tưởng tới mạch chạy được

Ba notebook đầu của `00_fundamental/` coi mạch lượng tử như một đối tượng toán học: viết `qc.h(0)`
thì có cổng H, viết `qc.ccx(0, 1, 2)` thì có Toffoli. Phần cứng thật không cho phép điều đó. Nó chỉ
biết vài cổng, chỉ cho phép cổng 2-qubit trên một số cặp qubit nhất định, và mỗi cổng đều có sai số.

**Compilation** (trong Qiskit gọi là **transpilation**) là bước dịch mạch lý tưởng thành mạch mà một
thiết bị cụ thể chạy được, sao cho càng ít sai số càng tốt. Đây là một bài toán **tối ưu**, và là
bài toán mà notebook này cùng cả folder `02_compilation/` tập trung vào.

Notebook này trả lời bốn câu:

1. Compile phải giải quyết những ràng buộc gì?
2. Qiskit chia công việc đó thành các **stage** nào?
3. Pass làm việc trên cấu trúc dữ liệu nào? (`DAGCircuit`, và khái niệm **front layer**)
4. Đo "mạch tốt" bằng hàm cost nào?

Câu 4 quan trọng nhất cho phần sau: muốn dùng reinforcement learning để compile thì **reward chính
là hàm cost này**. Chọn sai hàm cost thì agent học giỏi một việc vô nghĩa.

In [ ]:
import time

import numpy as np
import pandas as pd

from qiskit import QuantumCircuit, transpile
from qiskit.converters import circuit_to_dag
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import CouplingMap, generate_preset_pass_manager

import qiskit

print("qiskit:", qiskit.__version__)

# Thiết bị giả lập dùng xuyên suốt notebook: 5 qubit nối thành đường thẳng 0-1-2-3-4
COUPLING = CouplingMap.from_line(5)
BASIS = ["cx", "rz", "sx", "x"]

print("coupling map:", sorted(set(tuple(sorted(e)) for e in COUPLING.get_edges())))
print("basis gates :", BASIS)

## 1. Ba ràng buộc của phần cứng

**(a) Basis gates.** Thiết bị chỉ thực hiện được một tập cổng rất nhỏ. Ở đây là `cx`, `rz`, `sx`,
`x` - đủ để biểu diễn mọi unitary, nhưng mọi cổng khác phải được *dịch* ra tập này. Toffoli
(`ccx`) không có trong basis, nên nó sẽ nở ra thành một chuỗi dài.

**(b) Connectivity.** Cổng 2-qubit chỉ chạy được trên những cặp qubit nối trực tiếp với nhau.
Coupling map ở trên là một đường thẳng, nên `cx(0, 4)` là **không hợp lệ** - compiler phải chèn
SWAP để kéo hai qubit về cạnh nhau. Đây là bài toán **routing**, nội dung của notebook `02`.

**(c) Noise.** Mỗi cổng có xác suất lỗi. Cổng 2-qubit sai khoảng $10^{-3}$ tới $10^{-2}$, tức là
**một đến hai bậc tệ hơn** cổng 1-qubit. Nên hàm cost thực tế gần như luôn đặt trọng số lớn nhất
vào **số cổng 2-qubit**.

Xem ba ràng buộc đó làm gì với một mạch nhỏ:

In [ ]:
qc = QuantumCircuit(5)
qc.h(0)
qc.ccx(0, 1, 2)      # không có trong basis
qc.swap(1, 4)        # không có trong basis, lại còn trên cặp qubit không kề nhau
qc.ry(0.3, 3)        # không có trong basis
qc.cz(0, 4)          # qubit 0 và 4 cách nhau 4 bước trên đường thẳng
qc.ccx(2, 3, 4)
qc.measure_all()

sau = transpile(qc, basis_gates=BASIS, coupling_map=COUPLING,
                optimization_level=2, seed_transpiler=1)

print("TRƯỚC compile")
print("  cổng :", dict(qc.count_ops()))
print("  depth:", qc.depth(), "| size:", qc.size())
print()
print("SAU compile")
print("  cổng :", dict(sau.count_ops()))
print("  depth:", sau.depth(), "| size:", sau.size())
print()
print(f"-> depth tăng {sau.depth() / qc.depth():.1f} lần. Mạch bạn viết KHÔNG phải mạch được chạy.")

Mạch gốc có depth 6 và trông vô hại. Mạch thật chạy trên thiết bị có depth 40 và 19 cổng `cx`.
Toàn bộ khoảng cách đó do compiler quyết định - và đó là lý do người ta nghiên cứu compiler lượng tử:
mỗi `cx` tiết kiệm được là một phần sai số bớt đi.

## 2. Sáu stage của pass manager

Qiskit không compile bằng một hàm khổng lồ. Nó chạy một chuỗi **pass**, mỗi pass làm một việc nhỏ.
Chuỗi đó được nhóm thành sáu **stage** cố định:

| Stage | Việc | Pass tiêu biểu |
|---|---|---|
| `init` | chuẩn hoá đầu vào, tách cổng nhiều qubit | `UnitarySynthesis`, `HighLevelSynthesis` |
| `layout` | chọn qubit vật lý nào đóng vai qubit logic nào | `SabreLayout`, `VF2Layout` |
| `routing` | chèn SWAP cho mọi cổng 2-qubit thành hợp lệ | `SabreSwap`, `BasicSwap` |
| `translation` | dịch mọi cổng về đúng basis gates | `BasisTranslator` |
| `optimization` | huỷ cổng dư, gộp cổng, lặp tới khi không giảm nữa | `Optimize1qGatesDecomposition`, `CommutativeCancellation` |
| `scheduling` | gán thời điểm thực thi, chèn delay | `ALAPScheduleAnalysis`, `PadDelay` |

`generate_preset_pass_manager` dựng sẵn chuỗi này theo `optimization_level`. Kết quả là một
`StagedPassManager` - xem được từng stage có bao nhiêu pass:

In [ ]:
pm = generate_preset_pass_manager(optimization_level=2, basis_gates=BASIS,
                                  coupling_map=COUPLING, seed_transpiler=1)

print(type(pm).__name__, "- các stage:", pm.stages)
print()
tong = 0
for ten in pm.stages:
    stage = getattr(pm, ten, None)
    if stage is None:
        print(f"  {ten:14s} (rỗng)")
        continue
    task = list(stage.to_flow_controller().tasks)
    tong += len(task)
    print(f"  {ten:14s} {len(task)} task")
print(f"\n-> {tong} task ở mức stage, nhưng một task có thể là vòng lặp chứa nhiều pass bên trong.")

### Xem từng pass chạy và làm gì

`pm.run(qc, callback=...)` gọi callback sau **mỗi** pass, kèm DAG hiện tại. Đây là cách tốt nhất để
nhìn thấy mạch biến đổi dần - và cũng là cách thu metric nếu bạn muốn so sánh các pipeline.

In [ ]:
lich_su = []


def ghi_lai(**kw):
    """Callback của PassManager: được gọi sau mỗi pass."""
    dag = kw["dag"]
    lich_su.append({
        "stt": kw["count"],
        "pass": kw["pass_"].name(),
        "depth": dag.depth(),
        "size": dag.size(),
        "cx": dag.count_ops().get("cx", 0),
        "ms": round(kw["time"] * 1000, 2),
    })


_ = pm.run(qc, callback=ghi_lai)
df = pd.DataFrame(lich_su)

print(f"Tổng cộng {len(df)} pass đã chạy.\n")
# Chỉ in những pass THỰC SỰ làm đổi mạch
doi = df[(df["depth"].diff() != 0) | (df["size"].diff() != 0)]
print(doi.to_string(index=False))

Đọc bảng trên từ dưới lên là thấy rõ phân công:

- `BasisTranslator` là pass làm mạch **nở ra** nhiều nhất (dịch `ccx`, `ry`, `cz` ra basis).
- Các pass `optimization` sau đó **co** mạch lại.
- Phần lớn trong 47 pass không đổi gì cả - chúng là pass phân tích (`AnalysisPass`) chỉ ghi thông
  tin vào `property_set`, hoặc pass kiểm tra điều kiện dừng của vòng lặp optimization.

Phân biệt hai loại pass này là bắt buộc nếu sau này bạn tự viết pass (notebook `04`):

- **`AnalysisPass`** - đọc DAG, ghi kết quả vào `property_set`, **không** sửa DAG.
- **`TransformationPass`** - trả về DAG mới.

## 3. `DAGCircuit`: cấu trúc dữ liệu mà pass thật sự làm việc trên

`QuantumCircuit` là một **danh sách** cổng theo thứ tự bạn viết. Danh sách thì giả tạo: hai cổng
trên hai qubit khác nhau viết trước sau nhưng thực ra độc lập, chạy song song được.

Pass vì vậy làm việc trên `DAGCircuit` - một **đồ thị có hướng không chu trình**, trong đó cạnh là
qubit/clbit và đỉnh là cổng. Cạnh chính là quan hệ phụ thuộc thật: cổng B nối sau cổng A nghĩa là B
*phải* chờ A.

In [ ]:
dag = circuit_to_dag(qc)

print("số đỉnh cổng :", len(dag.op_nodes()))
print("cổng 2-qubit :", [n.name for n in dag.two_qubit_ops()])
print("cổng >2 qubit:", [n.name for n in dag.multi_qubit_ops() if len(n.qargs) > 2])
print("depth của DAG:", dag.depth())


def ten_qubit(dag, node):
    return [dag.find_bit(q).index for q in node.qargs]


print("\nthứ tự topo (10 cổng đầu):")
for n in list(dag.topological_op_nodes())[:10]:
    print(f"  {n.name:10s} qubit {ten_qubit(dag, n)}")

### Front layer - khái niệm quan trọng nhất của notebook này

**Front layer** là tập các cổng **không còn cổng nào chưa chạy ở phía trước nó**, tức là tập cổng
*chạy được ngay bây giờ*.

Mọi thuật toán routing đều hoạt động theo đúng một vòng lặp:

```
while DAG chưa rỗng:
    thực thi mọi cổng trong front layer mà coupling map cho phép
    nếu không thực thi được cổng nào:
        chèn một SWAP  <-- đây là chỗ duy nhất cần "quyết định"
```

Dòng `chèn một SWAP` là nơi mọi heuristic (và sau này là mọi RL policy) khác nhau. Front layer
chính là **state** của bài toán, và chọn SWAP nào chính là **action**. Notebook `05` sẽ biến đúng
vòng lặp này thành một `gymnasium.Env`.

Xem front layer tiến triển:

In [ ]:
dag2 = circuit_to_dag(qc)
print("Mô phỏng vòng lặp: cứ lấy cổng đầu của front layer rồi thực thi nó\n")

for buoc in range(6):
    fl = dag2.front_layer()
    if not fl:
        print("  DAG rỗng -> xong")
        break
    mo_ta = ", ".join(f"{n.name}{ten_qubit(dag2, n)}" for n in fl)
    print(f"  bước {buoc}: front layer ({len(fl)} cổng) = {mo_ta}")
    dag2.remove_op_node(fl[0])     # "thực thi" cổng đầu tiên

print(f"\n-> front layer thay đổi sau mỗi cổng được thực thi. Còn lại {dag2.size()} cổng.")

## 4. Hàm cost: "mạch tốt" nghĩa là gì?

Có bốn thước đo hay dùng, và chúng **không cùng chiều**:

| Thước đo | Qiskit | Ý nghĩa vật lý |
|---|---|---|
| `depth` | `qc.depth()` | thời gian chạy - so với thời gian decoherence $T_2$ |
| `size` | `qc.size()` | tổng số cổng |
| **số cổng 2-qubit** | `qc.count_ops()["cx"]` | **chi phối sai số**, trọng số lớn nhất |
| số SWAP | `count_ops()["swap"]` | giá phải trả cho connectivity (1 SWAP = 3 CX) |

Ước lượng thô nhưng cực kỳ hữu dụng: nếu mạch có $n$ cổng 2-qubit, mỗi cổng sai với xác suất
$\epsilon$, thì xác suất chạy trọn mạch không lỗi xấp xỉ

$$P_{\text{thành công}} \approx (1 - \epsilon)^{\,n} \approx e^{-\epsilon n}$$

Công thức này giải thích vì sao giảm 10% số CX đáng giá hơn giảm 10% số cổng 1-qubit rất nhiều, và
nó là ứng cử viên tự nhiên cho **reward** trong một bài toán RL.

In [ ]:
rows = []
for lv in range(4):
    t0 = time.time()
    out = transpile(qc, basis_gates=BASIS, coupling_map=COUPLING,
                    optimization_level=lv, seed_transpiler=1)
    ops = out.count_ops()
    rows.append({
        "optimization_level": lv,
        "depth": out.depth(),
        "size": out.size(),
        "cx": ops.get("cx", 0),
        "1q": sum(v for k, v in ops.items() if k in ("rz", "sx", "x")),
        "giây": round(time.time() - t0, 3),
    })

bang = pd.DataFrame(rows)
print(bang.to_string(index=False))
print()
print("-> level 0 -> 2 giảm được", rows[0]["cx"] - rows[2]["cx"], "cổng cx.")
print("-> level 3 không hơn level 2 ở mạch nhỏ này, nhưng tốn thời gian compile hơn.")

Giờ quy số cổng đó thành xác suất thành công, dùng **error rate thật** của từng cặp qubit. Qiskit có
`GenericBackendV2` - thiết bị giả lập nhưng có `Target` đầy đủ: basis gates, coupling map, **và
error rate riêng cho từng edge**.

`Target` chính là thứ cho phép compile **noise-aware**: không phải mọi cặp qubit đều tốt như nhau,
nên đặt cổng 2-qubit vào cặp tốt là một cách giảm sai số mà không cần giảm số cổng.

In [ ]:
backend = GenericBackendV2(num_qubits=5, coupling_map=COUPLING.get_edges(), seed=11)

loi_cx = {edge: props.error for edge, props in backend.target["cx"].items()}
print("error rate của cx trên từng edge:")
for edge in sorted(loi_cx):
    print(f"  cx{edge}: {loi_cx[edge]:.5f}")

tb = np.mean(list(loi_cx.values()))
print(f"\nerror trung bình: {tb:.5f}  (edge tệ nhất gấp {max(loi_cx.values())/min(loi_cx.values()):.1f} lần edge tốt nhất)")

print("\nXác suất chạy trọn mạch không lỗi, ước lượng chỉ bằng cổng cx:")
for lv in range(4):
    out = transpile(qc, backend=backend, optimization_level=lv, seed_transpiler=1)
    n_cx = out.count_ops().get("cx", 0)
    print(f"  level {lv}: {n_cx:3d} cx -> P ~ {(1 - tb) ** n_cx:.4f}")

Từ 0.89 lên 0.94 chỉ bằng cách đổi `optimization_level`. Đó là toàn bộ lý do compiler lượng tử là
một lĩnh vực nghiên cứu: cùng một thuật toán, cùng một thiết bị, chất lượng kết quả chênh nhau vì
compiler.

Và đó cũng là chỗ reinforcement learning chen vào được: các pass hiện tại dùng **heuristic do người
thiết kế**. Nếu một policy học được quyết định tốt hơn heuristic đó, mọi mạch chạy trên thiết bị đều
được lợi.

## Tóm tắt

| Khái niệm | Là gì | Dùng ở đâu tiếp |
|---|---|---|
| basis gates | tập cổng thiết bị chạy được | `03` - synthesis |
| coupling map | cặp qubit nào cho phép cổng 2-qubit | `01`, `02` |
| `Target` | basis + coupling + **error rate** | `01` - noise-aware layout, `05` - reward |
| stage | `init` → `layout` → `routing` → `translation` → `optimization` → `scheduling` | `04` - tự thay stage |
| `DAGCircuit` | IR dạng đồ thị, cạnh = phụ thuộc thật | mọi notebook sau |
| **front layer** | tập cổng chạy được ngay | `02`, **`05` - state của MDP** |
| hàm cost | depth, size, **số cổng 2-qubit**, số SWAP | `04` - benchmark, `05` - reward |
| `AnalysisPass` vs `TransformationPass` | đọc vs sửa DAG | `04` - tự viết pass |

### Thử tự làm

1. Đổi `COUPLING` thành `CouplingMap.from_ring(5)` (thêm một edge 4-0) rồi chạy lại ô so sánh
   `optimization_level`. Thêm **một** đường nối tiết kiệm được bao nhiêu `cx`?
2. Trong callback, thêm cột đếm `swap` thay vì `cx`. Pass nào tạo ra SWAP, pass nào xoá chúng?
3. Mạch trên có 2 cổng `ccx`. Thử `transpile` chỉ với `basis_gates=BASIS` mà **không** truyền
   `coupling_map`: bao nhiêu `cx` cho mỗi `ccx`? (Con số lý thuyết là 6.)